In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

train_dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=transform_train
)

test_dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=False,
    download=True,
    transform=transform_test
)

train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

test_loader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False
)

100%|██████████| 170M/170M [00:05<00:00, 33.0MB/s]


In [2]:
class CNN_Model(nn.Module):
    def __init__(self):
        super(CNN_Model, self).__init__()

        self.conv = nn.Sequential(

            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128*4*4, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 10)
        )

    def forward(self, x):
        x = self.conv(x)
        x = self.fc(x)
        return x

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = CNN_Model().to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [6]:
num_epochs = 50

for epoch in range(num_epochs):

    model.train()
    running_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:

        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs,1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    acc = 100 * correct / total

    print(f"Epoch {epoch+1} Loss:{running_loss:.3f} Accuracy:{acc:.2f}%")

Epoch 1 Loss:239.271 Accuracy:89.52%
Epoch 2 Loss:234.785 Accuracy:89.71%
Epoch 3 Loss:232.675 Accuracy:89.69%
Epoch 4 Loss:236.800 Accuracy:89.49%
Epoch 5 Loss:233.501 Accuracy:89.59%
Epoch 6 Loss:235.544 Accuracy:89.59%
Epoch 7 Loss:234.640 Accuracy:89.68%
Epoch 8 Loss:234.040 Accuracy:89.65%
Epoch 9 Loss:230.808 Accuracy:89.75%
Epoch 10 Loss:232.806 Accuracy:89.80%
Epoch 11 Loss:229.585 Accuracy:89.89%
Epoch 12 Loss:224.178 Accuracy:90.27%
Epoch 13 Loss:228.355 Accuracy:89.82%
Epoch 14 Loss:229.645 Accuracy:89.97%
Epoch 15 Loss:228.393 Accuracy:89.79%
Epoch 16 Loss:225.264 Accuracy:89.99%
Epoch 17 Loss:227.890 Accuracy:89.85%
Epoch 18 Loss:228.918 Accuracy:89.80%
Epoch 19 Loss:229.974 Accuracy:89.78%
Epoch 20 Loss:223.180 Accuracy:90.12%
Epoch 21 Loss:221.998 Accuracy:90.16%
Epoch 22 Loss:228.774 Accuracy:89.83%
Epoch 23 Loss:223.354 Accuracy:90.08%
Epoch 24 Loss:228.077 Accuracy:89.82%
Epoch 25 Loss:222.163 Accuracy:90.11%
Epoch 26 Loss:225.025 Accuracy:90.09%
Epoch 27 Loss:224.080

In [ ]:
#tong epoch 200